Url Connection & Data Cleaning

In [70]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 3

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
  master_df = pd.concat(dataframe_list, ignore_index=True)
  print("\n Collection complete!")
  print(f"Total rows collected across all locations: {len(master_df)}")
else:
  print("\n No data was collected. Check your API key or coordinates.")
master_df.head()


Successfully added 117 fire points from Amazon Rainforest.
Successfully added 116 fire points from California.
Successfully added 2745 fire points from Siberian Taiga.
Successfully added 32 fire points from New South Wales.
Successfully added 324 fire points from Pantanal.
Successfully added 78 fire points from Alberta.
Successfully added 1748 fire points from Mediterranean Basin.
Successfully added 561 fire points from Indonesia.
Successfully added 70 fire points from Greece.
Successfully added 988 fire points from Algeria.

 Collection complete!
Total rows collected across all locations: 6779


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,location_name
0,4.92096,-69.35458,304.01,0.38,0.36,2026-05-26,608,N20,VIIRS,n,2.0NRT,286.77,1.32,N,Amazon Rainforest
1,4.92372,-69.35061,313.87,0.38,0.36,2026-05-26,608,N20,VIIRS,n,2.0NRT,286.70,1.32,N,Amazon Rainforest
2,4.92426,-69.35406,322.44,0.38,0.36,2026-05-26,608,N20,VIIRS,n,2.0NRT,287.47,1.32,N,Amazon Rainforest
3,4.92481,-69.35751,315.58,0.38,0.36,2026-05-26,608,N20,VIIRS,n,2.0NRT,288.19,2.34,N,Amazon Rainforest
4,4.92648,-69.34664,303.37,0.38,0.36,2026-05-26,608,N20,VIIRS,n,2.0NRT,286.78,0.82,N,Amazon Rainforest
